# Download Pl@ntNet-300K subset (20 species)

Run once. Saves the images we use to `MyDrive/plantnet_dlnn26` as one zip file.

The full zip on Zenodo has 41.8 GB, so we only download the folders of our 20 species.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q remotezip

In [ ]:
import os
import time
import struct
import zlib
import zipfile
import requests
import pandas as pd
from remotezip import RemoteZip

drive_dir = '/content/drive/MyDrive/plantnet_dlnn26'
local_dir = '/content/plantnet20'
os.makedirs(drive_dir, exist_ok=True)

zenodo = 'https://zenodo.org/api/records/10419064/files'
zip_url = zenodo + '/images.zip/content'

# 20 species (10 pairs of the same genus), see README
species_ids = [4, 3, 112, 225, 84, 171, 200, 202, 190, 15, 11, 406, 47, 46, 60, 186, 35, 243, 310, 366]
organs = ['flower', 'leaf', 'fruit', 'habit', 'bark']
train_cap = 1000  # max train images per species
seed = 42

In [ ]:
# Zenodo only allows ~133 requests per minute, so retry if it fails
def get(url, headers=None, tries=6):
    for k in range(tries):
        try:
            r = requests.get(url, headers=headers or {}, timeout=600)
            if r.status_code == 429:
                time.sleep(60)
                continue
            r.raise_for_status()
            return r.content
        except requests.RequestException as e:
            if k == tries - 1:
                raise
            print('retrying', e)
            time.sleep(10 * (k + 1))

# Metadata files
for name in ['plantnet300K_metadata.csv', 'species_metadata.csv']:
    path = drive_dir + '/' + name
    if not os.path.exists(path):
        with open(path, 'wb') as f:
            f.write(get(zenodo + '/' + name + '/content'))

meta = pd.read_csv(drive_dir + '/plantnet300K_metadata.csv')
species = pd.read_csv(drive_dir + '/species_metadata.csv')

In [ ]:
# Keep our species and organs, and max 1000 train images per species
df = meta[meta['species_id'].isin(species_ids) & meta['organ'].isin(organs)]

train = df[df['split'] == 'train'].sample(frac=1, random_state=seed)
train = train[train.groupby('species_id').cumcount() < train_cap]

sel = pd.concat([train, df[df['split'] != 'train']])
sel = sel.sort_values(['split', 'species_id', 'PN_hash'])
sel['path'] = 'images/' + sel['split'] + '/' + sel['species_id'].map('{:04d}'.format) + '/' + sel['PN_hash'] + '.jpg'
sel = sel.merge(species[['species_id', 'species', 'genus']], on='species_id').reset_index(drop=True)
sel.to_csv(drive_dir + '/selection.csv', index=False)

print(len(sel))
print(sel['split'].value_counts())
sel.pivot_table(index='species', columns='split', values='PN_hash', aggfunc='count')

In [ ]:
# Read one file from the downloaded bytes
def read_file(chunk, base, info):
    o = info.header_offset - base
    n_name, n_extra = struct.unpack('<HH', chunk[o + 26:o + 30])
    start = o + 30 + n_name + n_extra
    data = chunk[start:start + info.compress_size]
    if info.compress_type == 8:
        data = zlib.decompressobj(-15).decompress(data)
    assert zlib.crc32(data) & 0xFFFFFFFF == info.CRC
    return data

# List of files inside the zip (only reads the index, not the images)
with RemoteZip(zip_url) as z:
    infos = sorted(z.infolist(), key=lambda i: i.header_offset)
pos = {info.filename: k for k, info in enumerate(infos)}

# Each split/species folder is one block in the zip, so one request per folder (60 in total)
wanted = set(sel['path'])
folders = sorted(set(p.rsplit('/', 1)[0] + '/' for p in wanted))

start_time = time.time()
count = 0
for i, folder in enumerate(folders):
    idx = [pos[n] for n in pos if n.startswith(folder) and not n.endswith('/')]
    first, last = min(idx), max(idx)
    base = infos[first].header_offset
    end = infos[last + 1].header_offset - 1
    t = time.time()
    chunk = get(zip_url, headers={'Range': 'bytes=%d-%d' % (base, end)})
    n = 0
    for k in range(first, last + 1):
        info = infos[k]
        if info.filename in wanted:
            out = os.path.join(local_dir, info.filename)
            os.makedirs(os.path.dirname(out), exist_ok=True)
            with open(out, 'wb') as f:
                f.write(read_file(chunk, base, info))
            n += 1
    count += n
    print(i + 1, folder, n, 'images', int(time.time() - t), 's')

print(count, 'images in', round((time.time() - start_time) / 60, 1), 'min')
assert count == len(sel)

In [ ]:
# Save as one zip in Drive (Drive is very slow with many small files)
zip_path = drive_dir + '/plantnet20_v1.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_STORED) as zf:
    for p in sel['path']:
        zf.write(os.path.join(local_dir, p), arcname=p)

with zipfile.ZipFile(zip_path) as zf:
    print(len(zf.namelist()), 'images in zip')
print(round(os.path.getsize(zip_path) / 1e9, 2), 'GB')

## Load the data in the other notebooks

If the folder was shared with you, first add a shortcut to it in your Drive (Organise > Add shortcut > My Drive).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp /content/drive/MyDrive/plantnet_dlnn26/plantnet20_v1.zip /content/
!unzip -q -o /content/plantnet20_v1.zip -d /content/plantnet20

import pandas as pd
sel = pd.read_csv('/content/drive/MyDrive/plantnet_dlnn26/selection.csv')
sel['file'] = '/content/plantnet20/' + sel['path']
sel.head()